In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

FILE_PATH = (
    "/content/drive/MyDrive/KB_FinalProject_Team6/"
    "완전쓰는거_ver2/인허가.csv"
)

# 코드와 ID를 보존하기 위해 우선 문자열로 불러오기
df_raw = pd.read_csv(
    FILE_PATH,
    encoding="utf-8-sig",
    dtype="string"
)

# 원본 보존용과 작업용 분리
df = df_raw.copy()

print(f"행 수: {df.shape[0]:,}")
print(f"컬럼 수: {df.shape[1]}")
display(df.head())

In [ ]:
column_report = pd.DataFrame({
    "자료형": df.dtypes.astype(str),
    "결측수": df.isna().sum(),
    "결측률(%)": df.isna().mean().mul(100).round(2),
    "고유값수": df.nunique(dropna=True)
})

display(column_report)

In [ ]:
print("완전히 동일한 중복 행:", df.duplicated().sum())
print("점포ID 결측:", df["점포ID"].isna().sum())
print("점포ID 중복:", df["점포ID"].duplicated().sum())
print("고유 점포이력ID:", df["점포이력ID"].nunique())

history_counts = df["점포이력ID"].value_counts()

print("인허가가 2건 이상 연결된 점포이력ID:",
      history_counts.gt(1).sum())

display(
    history_counts
    .rename_axis("점포이력ID")
    .reset_index(name="인허가건수")
    .head(10)
)

In [ ]:
display(
    df["업종명"]
    .value_counts(dropna=False)
    .rename_axis("업종명")
    .reset_index(name="인허가건수")
)

display(
    df[[
        "소재지면적",
        "시설면적",
        "면적_보완",
        "면적확인상태"
    ]].head(20)
)

display(df["면적확인상태"].value_counts(dropna=False))

In [ ]:
len([df.columns][0])

In [ ]:
# 앞으로 사용할 컬럼: 22개
keep_cols = [
    # 식별자
    "점포ID", "점포이력ID",

    # 업종
    "대분류", "업종명", "카드업종",
    "카드매핑상태", "업태구분명",

    # 지역·좌표
    "구군", "법정동", "행정동코드", "행정동명",
    "lon", "lat",

    # 개업일·점포명
    "인허가일자", "사업장명",

    # 면적 및 검증 정보
    "소재지면적", "시설면적", "면적_보완", "면적확인상태",

    # 프랜차이즈
    "프랜차이즈유무", "프랜차이즈_구분", "프랜차이즈_브랜드"
]

# 현재 작업 데이터에서 필요한 컬럼만 선택
# df_raw는 변경하지 않음
removed_cols = [col for col in df.columns if col not in keep_cols]
df = df.loc[:, keep_cols].copy()

print("제외한 컬럼:", removed_cols)
print(f"정리 후: {df.shape[0]:,}행 × {df.shape[1]}개 컬럼")

display(df.head())

In [ ]:
df.shape

In [ ]:
# 문자열 컬럼만 공백 정리
string_cols = df.select_dtypes(include=["string", "object"]).columns

for col in string_cols:
    df[col] = df[col].astype("string").str.strip()

    # 공백만 있던 값은 결측으로 처리
    df[col] = df[col].replace("", pd.NA)

# 변환 오류를 비교하기 위한 복사본
before_type_conversion = df.copy()

In [ ]:

# 숫자로 사용할 컬럼
numeric_cols = [
    "lon", "lat",
    "소재지면적", "시설면적", "면적_보완",
    "프랜차이즈유무"
]

for col in numeric_cols:
    # 숫자에 천 단위 쉼표가 있으면 제거
    values = (
        df[col]
        .astype("string")
        .str.replace(",", "", regex=False)
    )

    df[col] = pd.to_numeric(values, errors="coerce")

In [ ]:
# 변환 전 날짜로 복원
df["인허가일자"] = before_type_conversion["인허가일자"].copy()

print("인허가일자 결측 수:", df["인허가일자"].isna().sum())
display(df[["점포ID", "사업장명", "인허가일자"]].head(10))

In [ ]:
missing_date_rows = df.loc[df["인허가일자"].isna()].copy()

print(f"인허가일자 결측 행: {len(missing_date_rows):,}건")
display(missing_date_rows)

In [ ]:
# 원본 날짜 유지
# 현재까지 행 삭제·재정렬 없이 진행한 데이터 기준
assert df.index.equals(df_raw.index), "원본과 작업 데이터의 행 인덱스를 확인하세요."

df["인허가일자"] = df_raw["인허가일자"].copy()

date_text = df["인허가일자"].astype("string").str.strip()

# 시간까지 있는 형식으로 먼저 변환
date_parsed = pd.to_datetime(
    date_text,
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# 날짜만 있는 값도 추가로 처리
retry = date_text.notna() & date_parsed.isna()

date_parsed.loc[retry] = pd.to_datetime(
    date_text.loc[retry],
    format="%Y-%m-%d",
    errors="coerce"
)

# 분석용 컬럼 생성: 원본 컬럼은 변경하지 않음
df["인허가일자_dt"] = date_parsed

failed = date_text.notna() & df["인허가일자_dt"].isna()

print("원본 날짜 결측:", df["인허가일자"].isna().sum())
print("날짜 변환 실패:", failed.sum())
print("최초 인허가일:", df["인허가일자_dt"].min())
print("최근 인허가일:", df["인허가일자_dt"].max())

display(df[["인허가일자", "인허가일자_dt"]].head(10))

# 실패한 값이 있으면 원본 확인
if failed.any():
    display(
        df.loc[failed, ["점포ID", "사업장명", "인허가일자"]]
    )

In [ ]:
# 데이터 점검 기준일
check_date = pd.Timestamp("2026-10-28")

checks = {
    "인허가일자 결측": df["인허가일자_dt"].isna(),
    "기준일 이후 인허가": df["인허가일자_dt"] > check_date,

    "소재지면적 0 이하": df["소재지면적"] <= 0,
    "시설면적 0 이하": df["시설면적"] <= 0,
    "면적_보완 0 이하": df["면적_보완"] <= 0,

    "경도 결측": df["lon"].isna(),
    "위도 결측": df["lat"].isna(),
    "경도 허용범위 밖": ~df["lon"].between(-180, 180) & df["lon"].notna(),
    "위도 허용범위 밖": ~df["lat"].between(-90, 90) & df["lat"].notna(),
    "경도 또는 위도 0": (df["lon"] == 0) | (df["lat"] == 0),

    "프랜차이즈 0·1 이외": (
        df["프랜차이즈유무"].notna()
        & ~df["프랜차이즈유무"].isin([0, 1])
    )
}

quality_report = pd.DataFrame({
    "점검항목": list(checks),
    "건수": [int(mask.fillna(False).sum()) for mask in checks.values()]
})

display(quality_report)

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
keep_cols = [
    "점포ID",
    "점포이력ID",
    "대분류",
    "업종명",
    "카드업종",
    "업태구분명",
    "구군",
    "법정동",
    "행정동코드",
    "lon",
    "lat",
    "사업장명",
    "프랜차이즈유무",
    "프랜차이즈_구분",
    "프랜차이즈_브랜드",
    "인허가일자_dt"
]

# 누락 컬럼이 있으면 변경 전에 알려주기
missing_cols = [col for col in keep_cols if col not in df.columns]

if missing_cols:
    print("현재 df에 없는 컬럼:", missing_cols)
else:
    df = df.loc[:, keep_cols].copy()
    print(f"정리 완료: {df.shape[0]:,}행 × {df.shape[1]}개 컬럼")
    display(df.head())

In [ ]:
from pathlib import Path

keep_cols = [
    "점포ID",
    "점포이력ID",
    "대분류",
    "업종명",
    "카드업종",
    "업태구분명",
    "구군",
    "법정동",
    "행정동코드",
    "lon",
    "lat",
    "사업장명",
    "프랜차이즈유무",
    "프랜차이즈_구분",
    "프랜차이즈_브랜드",
    "인허가일자_dt"
]

# 필요한 컬럼이 모두 있는지 확인
missing_cols = [col for col in keep_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"누락된 컬럼: {missing_cols}")

# 기본 데이터 확정
df_base = df.loc[:, keep_cols].copy()

# 원본과 다른 결과 폴더에 저장
save_dir = Path(
    "/content/drive/MyDrive/KB_FinalProject_Team6/인허가_분석결과"
)
save_dir.mkdir(parents=True, exist_ok=True)

# 재실행 시 기존 결과를 덮어쓰지 않도록 시간 표시
stamp = pd.Timestamp.now(tz="Asia/Seoul").strftime("%Y%m%d_%H%M%S")
file_stem = f"인허가_기본16컬럼_{stamp}"

# CSV: 엑셀 등에서 확인하기 편한 파일
df_base.to_csv(
    save_dir / f"{file_stem}.csv",
    index=False,
    encoding="utf-8-sig"
)

# Parquet: 날짜·숫자·문자열 자료형을 유지하는 분석용 파일
df_base.to_parquet(
    save_dir / f"{file_stem}.parquet",
    index=False
)

# 이후 파생변수를 추가할 작업용 데이터
df = df_base.copy()

print(f"저장 완료: {df_base.shape[0]:,}행 × {df_base.shape[1]}개 컬럼")
print("저장 폴더:", save_dir)
print("파일명:", file_stem)

In [ ]:
df = pd.read_parquet(save_dir / f"{file_stem}.parquet")